# Auditoría COT ↔ OC, agosto 2026 — vínculo oficial

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Auditoria_COT_OC_Corregida.ipynb)

Notebook **solo lectura** del clon. Requiere `vinculos_COT_OC_2026-08_seleccionadas.csv`, derivado del campo oficial `CodigoOC` presente en `COT_2026-08.zip`. Se esperan 65.945 COT seleccionadas y 68.093 vínculos COT–OC. Una cotización puede tener más de una OC. Los archivos de OC de agosto no necesariamente incluyen órdenes de cotizaciones publicadas en agosto, por diferencias de período. No se infiere adjudicación definitiva de estados no finales.

El CSV de vínculos fue generado por separado y **no contiene necesariamente información verificable solo desde las pestañas del clon**. La auditoría no escribe en Sheets ni modifica el frontend o los ETL.

In [ ]:
!pip -q install gspread google-auth
from google.colab import auth,files
from google.auth import default
import gspread,csv,collections,json,unicodedata,re,datetime,decimal
auth.authenticate_user()
cred,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
gc=gspread.authorize(cred)
CLON='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
book=gc.open_by_key(CLON)
print('Conectado al clon en modo solo lectura')

In [ ]:
print('Sube vinculos_COT_OC_2026-08_seleccionadas.csv')
subidos=files.upload()
assert len(subidos)==1
nombre=next(iter(subidos))
with open(nombre,encoding='utf-8-sig',newline='') as f: links=list(csv.DictReader(f))
necesarias={'codigo_cot','codigo_oc','proveedor_seleccionado','rut_proveedor','monto_oferta_clp','estado_oc_historico','fecha_aceptacion_oc'}
assert links and necesarias.issubset(links[0].keys()),'El CSV no contiene los encabezados esperados'
assert len(links)==len({(r['codigo_cot'],r['codigo_oc']) for r in links}),'Vínculos repetidos'
print('Vínculos:',len(links),'Cotizaciones:',len({r['codigo_cot'] for r in links}))

In [ ]:
hr=['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc']
ho=['codigo_oc','codigo_proceso','fecha_creacion','fecha_aceptacion','estado','proveedor','monto_total_oc_clp','tipo_oc','cantidad_lineas','total_lineas_neto']
def leer(titulo,encabezados,lote=5000):
 ws=book.worksheet(titulo)
 assert ws.row_values(1)==encabezados,'Cabecera inesperada en '+titulo
 salida=[]
 for i in range(2,ws.row_count+1,lote):
  bloque=ws.get(f'A{i}:J{min(i+lote-1,ws.row_count)}',value_render_option='UNFORMATTED_VALUE')
  for fila in bloque:
   if not fila or not any(str(v).strip() for v in fila):continue
   fila=[str(v).strip() if v is not None else '' for v in fila]+['']*len(encabezados)
   salida.append(dict(zip(encabezados,fila)))
  if (i-2)//lote%5==0:print(titulo,len(salida))
 return salida
resultados=leer('Resultados',hr)
ordenes=leer('OrdenesCompra',ho)
print('Resultados:',len(resultados),'OC:',len(ordenes))

In [ ]:
def normal(s):
 s=unicodedata.normalize('NFKD',str(s or '').upper())
 return re.sub(r'[^A-Z0-9]','', ''.join(x for x in s if not unicodedata.combining(x)))
def dinero(s):
 try:return decimal.Decimal(str(s))
 except (decimal.InvalidOperation,ValueError):return None
oc_by={o['codigo_oc'].strip().upper():o for o in ordenes if o['codigo_oc'].strip()}
r_by={r['codigo'].strip().upper():r for r in resultados if r['codigo'].strip()}
clas=collections.Counter();cuenta_cot=collections.Counter();detalle=[]
for v in links:
 cot=v['codigo_cot'].strip().upper();codoc=v['codigo_oc'].strip().upper()
 cuenta_cot[cot]+=1
 resultado=r_by.get(cot);oc=oc_by.get(codoc)
 if resultado is None:categoria='COT_SIN_RESULTADO_EN_HOJA'
 elif oc is None:categoria='OC_NO_PRESENTE_EN_HISTORICO_AGOSTO'
 elif normal(oc['proveedor'])!=normal(v['proveedor_seleccionado']):categoria='CODIGO_OC_COINCIDE_PROVEEDOR_DISTINTO'
 elif dinero(oc['monto_total_oc_clp'])==dinero(v['monto_oferta_clp']):categoria='CODIGO_OC_PROVEEDOR_MONTO_COINCIDEN'
 else:categoria='CODIGO_OC_PROVEEDOR_COINCIDEN_MONTO_DIFERENTE'
 clas[categoria]+=1
 detalle.append({'codigo_cot':cot,'codigo_oc':codoc,'categoria':categoria,
  'proveedor_seleccionado':v['proveedor_seleccionado'],'rut_proveedor_cot':v['rut_proveedor'],
  'proveedor_oc':oc['proveedor'] if oc else '', 'monto_oferta_clp':v['monto_oferta_clp'],
  'monto_oc_clp':oc['monto_total_oc_clp'] if oc else '',
  'estado_oc_desde_cot':v['estado_oc_historico'],
  'estado_oc_desde_hoja':oc['estado'] if oc else '',
  'fecha_aceptacion_oc_cot':v['fecha_aceptacion_oc'],
  'fecha_aceptacion_oc_hoja':oc['fecha_aceptacion'] if oc else ''})
resumen={'fecha_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'cot_seleccionadas_en_vinculos':len(cuenta_cot),'vinculos_cot_oc':len(links),
 'cot_con_multiples_oc':sum(n>1 for n in cuenta_cot.values()),
 'max_oc_por_cot':max(cuenta_cot.values()),
 'resultados_en_hoja':len(resultados),'oc_en_hoja':len(ordenes),'categorias':dict(clas),
 'advertencia':'Coincidencia por CodigoOC oficial: no demuestra adjudicación formal; diferencia de cobertura temporal'}
print(json.dumps(resumen,ensure_ascii=False,indent=2))

In [ ]:
base='/content/auditoria_cot_oc_corregida_2026_08'
with open(base+'_resumen.json','w',encoding='utf-8') as f:json.dump(resumen,f,ensure_ascii=False,indent=2)
with open(base+'_detalle.csv','w',encoding='utf-8-sig',newline='') as f:
 w=csv.DictWriter(f,fieldnames=list(detalle[0]));w.writeheader();w.writerows(detalle)
files.download(base+'_resumen.json')
files.download(base+'_detalle.csv')
print('Archivos de auditoría preparados; Google Sheets no fue modificado')